## Step 1 — Parse PDF with Docling

In [1]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from image_handler import get_pipeline_options_with_images

import json
from pathlib import Path

# Use the helper that enables image materialisation
pipeline_options = get_pipeline_options_with_images(num_threads=4)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

PDF_PATH = "./test1.pdf"  
PAPER_ID = "cs301_jan2023" 

CACHE_PATH = Path("./cache/rbb3013_sept2025.json")

if CACHE_PATH.exists():
    # Load from cache — instant
    from docling_core.types.doc import DoclingDocument
    doc = DoclingDocument.model_validate_json(CACHE_PATH.read_text())
    print("Loaded from cache")
else:
    result = converter.convert(PDF_PATH)
    doc = result.document
    CACHE_PATH.parent.mkdir(exist_ok=True)
    CACHE_PATH.write_text(doc.model_dump_json())
    print("Parsed and cached")


print(f"Pages:    {len(doc.pages)}")
print(f"Texts:    {len(doc.texts)}")
print(f"Tables:   {len(doc.tables)}")
print(f"Pictures: {len(doc.pictures)}")

Loaded from cache
Pages:    10
Texts:    230
Tables:   4
Pictures: 6


## Step 2 — Chunk by question boundary

In [2]:
from question_chunker import chunk_questions

chunks = chunk_questions(doc)

print(f"Total chunks: {len(chunks)}\n")
for c in chunks:
    print(c)

[chunker] Processing pages 2–7 (skipping cover p1, appendix p8+)
[chunker] Detected 5 question(s): Q1, Q2, Q2, Q3, Q4
Total chunks: 5

QuestionChunk(q='1', level='main', marks=10, pages=[3], has_image=False, text='FIGURE Q1b Determine the set of series  resistances,  R1, Rz,  Ri and  R4,  requ...')
QuestionChunk(q='2', level='main', marks=None, pages=[4], has_image=False, text='2 a. FIGURE Q2a shows a pressured vessel level measurement. Solve the Lower Rang...')
QuestionChunk(q='2', level='main', marks=10, pages=[4, 5], has_image=False, text='FIGURE Q2a Gas pressure (Pgx) bugrsueduoo teg(mel) Density = Y: iguid pensity [1...')
QuestionChunk(q='3', level='main', marks=10, pages=[6, 7], has_image=False, text='FIGURE Q3 shows an instrument loop diagram for a flow control loop. Di~CIJSS the...')
QuestionChunk(q='4', level='main', marks=None, pages=[7], has_image=False, text='FIGURE Q4 D(s) Gjs) SP(s) ·+ E(s) MV(s)- CV(s) Gis) , Gp(s) •.•....')


In [4]:
# Inspect a single chunk in detail
IDX = 0  # change this
c = chunks[IDX]
print(f"Question: {c.question_number}")
print(f"Pages:    {c.page_numbers}")
print(f"Marks:    {c.marks}")
print(f"Has img:  {c.has_image}")
print(f"Type:     {c.question_type}")
print("---")
print(c.question_text)

Question: 1
Pages:    [3]
Marks:    10
Has img:  False
Type:     calculation
---
FIGURE Q1b
Determine the set of series  resistances,  R1, Rz,  Ri and  R4,  required  for this design.
[10 marks].
If  the  nleter ist.o b~ fu~her scaled .to  cater for 5..000 V  range,  deterfnillf:l t!:Je additionalshunt re!sistance, Rs,.·req~ired ifthEiseled~r is  positianed  orfthe 1000 V range selector switch.  Explain the effect of the other range selection points.
-"h :"
[5 marks]
ii.


## Step 3 — Extract & link images

In [ ]:
from image_handler import extract_and_link_images

# Save locally (change use_supabase=True when you're ready to deploy)
chunks = extract_and_link_images(
    doc,
    chunks,
    output_dir="./extracted_images",
    paper_id=PAPER_ID,
    use_supabase=False,
)

# Show which chunks have images
img_chunks = [c for c in chunks if c.has_image]
print(f"{len(img_chunks)} chunks have images")
for c in img_chunks:
    print(f"  Q{c.question_number}: {c.image_refs}")

In [ ]:
# Quick visual check — display an extracted image
from PIL import Image
import matplotlib.pyplot as plt

if img_chunks:
    img_path = img_chunks[0].image_refs[0]
    img = Image.open(img_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Q{img_chunks[0].question_number} — extracted image")
    plt.show()
else:
    print("No images found in this paper.")

## Step 4 — Validate before moving on

In [ ]:
# Sanity-check summary — run this to confirm quality before proceeding
import json

print("=== Chunk summary ===")
print(f"Total chunks:         {len(chunks)}")
print(f"With marks detected:  {sum(1 for c in chunks if c.marks)}")
print(f"With images:          {sum(1 for c in chunks if c.has_image)}")
print(f"Empty text (bad!):    {sum(1 for c in chunks if not c.question_text.strip())}")

print("\n=== Type breakdown ===")
from collections import Counter
types = Counter(c.question_type for c in chunks)
for t, count in types.items():
    print(f"  {t}: {count}")

print("\n=== First 5 chunks as JSON ===")
print(json.dumps([c.to_dict() for c in chunks[:5]], indent=2))

## ✅ Done — ready for next step

If the chunk summary looks correct (questions detected, marks parsed, images linked),
proceed to `02_Enrichment.ipynb` for:
- LLM topic tagging
- Contextual preamble generation
- Embedding (Ollama dense + BM25 sparse)

**If chunks look wrong** (too few, splits in wrong places), check:
1. Print `doc.texts` raw to see what Docling extracted
2. Adjust regex patterns in `question_chunker.py` to match your paper's format
3. Try `split_subquestions=False` to only split on main question numbers